# 01 — Tujuan dan Desain Eksperimen

Smoke test **YOLO26n baseline → LAMP structured channel pruning → save/reload → evaluasi**.
Input adalah checkpoint baseline W&B yang telah dilatih. Output adalah model dengan
tensor yang benar-benar lebih kecil, laporan audit, dan metrik sebelum/sesudah.
**Tidak ada recovery, fine-tuning, training, teacher, KD, backward, atau optimizer step.**
Penurunan mAP tanpa recovery dapat terjadi dan bukan alasan untuk menyembunyikan kegagalan.

LAMP asli (Lee et al., ICLR 2021) menormalisasi magnitude kuadrat bobot yang diurutkan:
$s_i=w_i^2/\sum_{j\ge i}w_j^2$. Di sini `LAMPImportance` Torch-Pruning mereduksi
magnitude pada channel yang terikat dalam dependency group, lalu menerapkan normalisasi LAMP.
Ini **adaptasi structured channel pruning**, bukan replikasi unstructured masking paper asli.
Global berarti ranking di seluruh **group eligible**, bukan seluruh channel model.

**Run All:** aktifkan Internet + GPU di Kaggle, tambahkan dataset baseline, aktifkan secret
`WANDB_API_KEY` (atau session W&B yang sudah login), sesuaikan cell konfigurasi, lalu Run All.
Checkpoint lokal hanya untuk salinan artifact yang sama. Tidak ada download pengganti COCO/dataset.
Notebook tidak mengimpor package penelitian dari repository.

## 02 — Konfigurasi Eksperimen
Ubah parameter hanya di cell berikut. `CHECKPOINT_SELECTOR="best.pt"` adalah pilihan
eksplisit basename yang harus unik meskipun artifact memiliki `last.pt`/checkpoint lain.
Gunakan path relatif lengkap jika terdapat beberapa `best.pt`; tanpa selector, >1 kandidat ditolak.
`v84` adalah versi artifact, **bukan nomor epoch**. Rasio 10% bukan jaminan reduksi parameter 10%.

In [ ]:
from pathlib import Path

WANDB_ARTIFACT = (
    "ilhamrafiqin-brawijaya-university/yolom26-kd-indotraffic/"
    "03_baseline_yolo26n-9uu3meve-checkpoints:v84"
)
LOCAL_CHECKPOINT_PATH = None  # File .pt atau direktori hasil download artifact yang sama.
LOCAL_EXPECTED_SHA256 = None  # Opsional: SHA-256 dari provenance download sebelumnya.
CHECKPOINT_SELECTOR = "best.pt"  # None: hanya terima satu kandidat yang bernama best.pt.
DATASET_ROOT = Path("/kaggle/input/datasets/ilhamrafiqin/poltrada-v3-yolo26-clean")
DATA_YAML = DATASET_ROOT / "data.yaml"
OUTPUT_DIR = Path("/kaggle/working/lamp_yolo26n") if Path("/kaggle").exists() else Path("outputs/lamp_yolo26n")
PRUNING_RATIO = 0.10
GLOBAL_PRUNING = True
ITERATIVE_STEPS = 1
MAX_PRUNING_RATIO = 0.50
ROUND_TO = None
SEED = 42
DEVICE = "auto"  # auto: cuda:0 bila tersedia; alternatif: cpu, cuda:0.
RUN_FULL_VALIDATION = True
IMGSZ = 640
IMPORTANCE_P = 2
BATCH = 16
WORKERS = 2
INSTALL_DEPENDENCIES = True
WANDB_TRACKING = True  # Boleh False untuk mode checkpoint lokal tanpa login W&B.
BENCHMARK_LATENCY = True  # Hanya CUDA; CPU dilaporkan NOT_RUN.
LATENCY_WARMUP = 50
LATENCY_REPEATS = 200
RELOAD_TIMEOUT_SECONDS = 300
RECOVERY_ENABLED = False
KD_ENABLED = False
FINETUNING_ENABLED = False

In [ ]:
import csv
import hashlib
import importlib.metadata as metadata
import inspect
import json
import math
import os
import platform
import random
import subprocess
import sys
import time
import traceback
import uuid
import warnings
from contextlib import contextmanager
from copy import deepcopy
from datetime import datetime, timezone
from types import SimpleNamespace

PINNED = {"ultralytics": "8.4.155", "wandb": "0.28.1", "torch-pruning": "1.6.1"}
REQUIRED_CHECKS = (
    "baseline_loaded", "baseline_forward", "dependency_graph", "structural_channels_removed",
    "parameters_reduced", "pruned_forward", "protected_head_unchanged", "checkpoint_saved",
    "checkpoint_reloaded", "reload_consistent", "detection_prediction",
)


def write_json(path, value):
    """Tulis JSON standar; NaN/Inf ditolak, bukan disamarkan sebagai metrik."""
    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")


def sha256_file(path):
    """Hash berkas tanpa membaca seluruh checkpoint ke RAM sekaligus."""
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def new_state():
    """Status terpisah untuk validitas struktur dan evaluasi dataset."""
    return {"structural_smoke_status": "NOT_RUN", "validation_status": "NOT_RUN",
            "checks": {key: False for key in REQUIRED_CHECKS}, "errors": [], "warnings": [],
            "environment": {}, "provenance": {}, "manifest": {}, "metrics": {}, "stages": {}}


def flush_reports():
    """Persist laporan parsial agar kegagalan tidak hilang ketika Run All berhenti."""
    STATE["manifest"].update({
        "source": STATE["provenance"], "versions": STATE["environment"].get("packages", {}),
        "seed": SEED, "target_pruning_ratio": PRUNING_RATIO,
        "checks": STATE["checks"], "structural_smoke_status": STATE["structural_smoke_status"],
        "validation_status": STATE["validation_status"],
        "errors": STATE["errors"], "warnings": STATE["warnings"],
    })
    for filename, value in (
        ("environment.json", STATE["environment"]), ("checkpoint_provenance.json", STATE["provenance"]),
        ("pruning_manifest.json", STATE["manifest"]), ("before_after_metrics.json", STATE["metrics"]),
        ("smoke_test_report.json", {k: v for k, v in STATE.items()
                                    if k not in {"environment", "provenance", "manifest", "metrics"}}),
    ):
        write_json(RUN_DIR / filename, value)


@contextmanager
def checked_stage(name, *, validation=False):
    """Catat kegagalan, tutup run W&B, lalu teruskan exception asli."""
    STATE["stages"][name] = "RUNNING"
    flush_reports()
    try:
        yield
    except Exception as error:
        STATE["stages"][name] = "FAIL"
        STATE["errors"].append({"stage": name, "type": type(error).__name__,
                                "message": str(error), "traceback": traceback.format_exc()})
        STATE["validation_status" if validation else "structural_smoke_status"] = "FAIL"
        flush_reports()
        if WB_RUN is not None:
            WB_RUN.finish(exit_code=1)
        raise
    else:
        STATE["stages"][name] = "PASS"
        flush_reports()


def configuration():
    """Whitelist parameter publik; kredensial tidak masuk laporan."""
    keys = ("WANDB_ARTIFACT LOCAL_CHECKPOINT_PATH LOCAL_EXPECTED_SHA256 CHECKPOINT_SELECTOR DATASET_ROOT "
            "DATA_YAML OUTPUT_DIR PRUNING_RATIO GLOBAL_PRUNING ITERATIVE_STEPS MAX_PRUNING_RATIO ROUND_TO "
            "SEED DEVICE RUN_FULL_VALIDATION IMGSZ IMPORTANCE_P BATCH WORKERS INSTALL_DEPENDENCIES "
            "WANDB_TRACKING BENCHMARK_LATENCY LATENCY_WARMUP LATENCY_REPEATS RELOAD_TIMEOUT_SECONDS "
            "RECOVERY_ENABLED KD_ENABLED FINETUNING_ENABLED").split()
    return {key: str(globals()[key]) if isinstance(globals()[key], Path) else globals()[key] for key in keys}


def validate_configuration():
    """Tolak konfigurasi yang tidak termasuk eksperimen tanpa recovery."""
    if any((RECOVERY_ENABLED, KD_ENABLED, FINETUNING_ENABLED)):
        raise ValueError("Recovery, KD, dan fine-tuning wajib disabled.")
    if not 0 < PRUNING_RATIO <= MAX_PRUNING_RATIO <= 0.5:
        raise ValueError("Wajib 0 < PRUNING_RATIO <= MAX_PRUNING_RATIO <= 0.50.")
    if ITERATIVE_STEPS != 1 or IMPORTANCE_P != 2:
        raise ValueError("Smoke test ini hanya mendukung satu langkah dan importance p=2.")
    if IMGSZ < 32 or IMGSZ % 32 or BATCH < 1 or WORKERS < 0:
        raise ValueError("IMGSZ harus kelipatan 32, BATCH positif, WORKERS nonnegatif.")
    if ROUND_TO is not None and (not isinstance(ROUND_TO, int) or ROUND_TO < 1):
        raise ValueError("ROUND_TO harus None atau integer positif.")
    if WANDB_ARTIFACT.count("/") != 2 or ":" not in WANDB_ARTIFACT:
        raise ValueError("WANDB_ARTIFACT wajib entity/project/artifact:version.")

In [ ]:
RUN_DIR = Path(OUTPUT_DIR).expanduser().resolve() / (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
)
RUN_DIR.mkdir(parents=True, exist_ok=False)
STATE = new_state()
WB_RUN = None
write_json(RUN_DIR / "config.json", configuration())
with checked_stage("configuration"):
    validate_configuration()
print("Output run baru:", RUN_DIR)

## 03 — Pemeriksaan Environment dan Dependency
Versi penelitian dikunci, tetapi Torch/torchvision yang sudah terpasang dipertahankan
menggunakan pip constraints. Instalasi gagal bila resolver memerlukan penggantian keduanya.
Importance memakai implementasi paket terpasang; signature, versi metadata, dan hash source dicatat.
Autograd hanya dipakai untuk tracing Dependency Graph; seluruh modul tetap `eval()`.

In [ ]:
def ensure_dependencies(output_dir, install=True):
    """Pasang dependency tambahan sambil mengunci Torch/torchvision environment."""
    if sys.version_info < (3, 11):
        raise RuntimeError("Gunakan Python >=3.11.")
    preserved = {name: metadata.version(name) for name in ("torch", "torchvision")}
    needed = []
    for name, expected in PINNED.items():
        try:
            found = metadata.version(name)
        except metadata.PackageNotFoundError:
            found = None
        if found != expected:
            needed.append(f"{name}=={expected}")
    if needed and not install:
        raise RuntimeError(f"Dependency belum cocok: {needed}; aktifkan INSTALL_DEPENDENCIES.")
    if needed:
        constraints = Path(output_dir) / "preserve_torch.txt"
        constraints.write_text("\n".join(f"{k}=={v}" for k, v in preserved.items()) + "\n", encoding="utf-8")
        subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                        "--constraint", str(constraints), *needed], check=True)
    actual = {name: metadata.version(name) for name in preserved}
    if actual != preserved:
        raise RuntimeError(f"Torch/torchvision berubah: {preserved} -> {actual}; restart runtime asli.")
    for name, expected in PINNED.items():
        if metadata.version(name) != expected:
            raise RuntimeError(f"Versi {name} tidak sesuai pin {expected}; restart kernel bila perlu.")

In [ ]:
with checked_stage("dependency_installation"):
    ensure_dependencies(RUN_DIR, install=INSTALL_DEPENDENCIES)

In [ ]:
with checked_stage("dependency_imports"):
    import numpy as np
    import pandas as pd
    import torch
    import torch_pruning as tp
    import ultralytics
    import wandb
    import yaml
    from PIL import Image
    from ultralytics import YOLO
    from ultralytics.data.utils import check_det_dataset
    from ultralytics.nn.modules import Bottleneck, C2PSA, Detect
    from ultralytics.nn.modules.block import AAttn, Attention
    from ultralytics.nn.tasks import DetectionModel
    from ultralytics.utils.torch_utils import _attention_ops
    from thop import profile as thop_profile

In [ ]:
def seed_environment(seed):
    """Samakan RNG dan backend FP32 di notebook dan proses restore."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.set_float32_matmul_precision("highest")


def environment_report():
    """Periksa API aktual, bukan API yang diasumsikan dari contoh YOLOv8."""
    for name, expected in PINNED.items():
        if metadata.version(name) != expected:
            raise RuntimeError(f"Pin {name} tidak terpenuhi.")
    if ultralytics.__version__ != PINNED["ultralytics"]:
        raise RuntimeError("Modul Ultralytics lama masih di kernel; restart runtime.")
    signature = inspect.signature(tp.pruner.BasePruner)
    for key in ("global_pruning", "max_pruning_ratio", "round_to", "output_transform", "forward_fn"):
        if key not in signature.parameters:
            raise RuntimeError(f"BasePruner tidak mendukung {key}.")
    return {
        "python": sys.version, "platform": platform.platform(),
        "packages": {name: metadata.version(name) for name in (*PINNED, "torch", "torchvision", "numpy", "ultralytics-thop")},
        "cuda_available": torch.cuda.is_available(), "cuda_build": torch.version.cuda,
        "device": str(TARGET_DEVICE),
        "gpu": torch.cuda.get_device_name(TARGET_DEVICE) if TARGET_DEVICE.type == "cuda" else None,
        "precision": "FP32", "tf32": False, "cudnn_benchmark": False,
        "base_pruner_signature": str(signature),
        "lamp_signature": str(inspect.signature(tp.importance.LAMPImportance)),
        "lamp_source_sha256": hashlib.sha256(inspect.getsource(tp.importance.LAMPImportance).encode()).hexdigest(),
        "importance_module_sha256": sha256_file(inspect.getfile(tp.importance.LAMPImportance)),
        "flops_method": "THOP full-resolution Conv/BN + separate Ultralytics attention matmul counter; GFLOPs=2*MACs/1e9",
        "thop_profile_signature": str(inspect.signature(thop_profile)),
        "thop_distribution_versions": thop_versions(),
    }


def thop_versions():
    """Catat kedua distribution yang dapat menyediakan namespace thop yang sama."""
    result = {}
    for name in ("thop", "ultralytics-thop"):
        try:
            result[name] = metadata.version(name)
        except metadata.PackageNotFoundError:
            result[name] = None
    return result

In [ ]:
with checked_stage("environment"):
    TARGET_DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu") if DEVICE == "auto" else torch.device(DEVICE)
    if TARGET_DEVICE.type == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA diminta tetapi tidak tersedia.")
    seed_environment(SEED)
    STATE["environment"] = environment_report()
    print(json.dumps(STATE["environment"], indent=2))

## 04 — Mengambil Checkpoint dari W&B
Kandidat ditampilkan lengkap. Login memakai environment/session atau Kaggle Secrets;
nilai API key tidak ditulis ke konfigurasi/laporan. Mode lokal tidak mengunduh bobot lain.
Tanpa hash/provenance pembanding, asal salinan lokal dilaporkan **user_asserted**, bukan verified.

In [ ]:
def select_checkpoint(root, selector="best.pt"):
    """Pilih checkpoint unik secara eksplisit; tidak menebak dari urutan filesystem."""
    root = Path(root).expanduser().resolve()
    if root.is_file():
        candidates = [root] if root.suffix.lower() == ".pt" else []
        base = root.parent
    else:
        candidates = sorted(p.resolve() for p in root.rglob("*.pt") if p.is_file())
        base = root
    print("Kandidat checkpoint:", [p.relative_to(base).as_posix() for p in candidates])
    if not candidates:
        raise FileNotFoundError(f"Tidak ada file .pt di {root}.")
    if root.is_file():
        return root  # LOCAL_CHECKPOINT_PATH berupa file adalah pilihan eksplisit.
    if selector:
        selector = str(selector).replace("\\", "/")
        matches = [p for p in candidates if p.relative_to(base).as_posix() == selector] if "/" in selector else [p for p in candidates if p.name == selector]
        if len(matches) == 1:
            return matches[0]
        raise ValueError(f"Selector {selector!r} memiliki {len(matches)} kecocokan; pilih path relatif tepat.")
    if len(candidates) == 1 and candidates[0].name == "best.pt":
        return candidates[0]
    raise ValueError("Checkpoint ambigu atau best.pt tidak tersedia; isi CHECKPOINT_SELECTOR secara eksplisit.")


def checkpoint_provenance(path, artifact_name, *, expected_hash=None, resolved_artifact=None):
    """Catat identitas sumber; hash lokal opsional wajib cocok bila diberikan."""
    digest = sha256_file(path)
    if expected_hash is not None and digest.lower() != expected_hash.lower():
        raise ValueError("SHA-256 checkpoint lokal tidak cocok dengan LOCAL_EXPECTED_SHA256.")
    return {"requested_artifact": artifact_name, "resolved_artifact": resolved_artifact,
            "checkpoint_name": Path(path).name, "checkpoint_path": str(Path(path).resolve()),
            "sha256": digest, "file_bytes": Path(path).stat().st_size,
            "source_verification": "wandb_download" if resolved_artifact else ("expected_sha256_matched" if expected_hash else "user_asserted_local_copy")}


def obtain_checkpoint():
    """Unduh hanya artifact target atau pakai salinannya yang dipilih pengguna."""
    global WB_RUN
    if LOCAL_CHECKPOINT_PATH is None and not WANDB_TRACKING:
        raise ValueError("Download artifact memakai run.use_artifact; aktifkan WANDB_TRACKING atau pilih salinan lokal.")
    if WANDB_TRACKING:
        if "WANDB_API_KEY" not in os.environ and Path("/kaggle").exists() and not wandb.api.api_key:
            from kaggle_secrets import UserSecretsClient
            key = UserSecretsClient().get_secret("WANDB_API_KEY")
            wandb.login(key=key, relogin=False)
            del key
        entity, project, _ = WANDB_ARTIFACT.split("/", 2)
        WB_RUN = wandb.init(entity=entity, project=project, job_type="lamp-pruning-smoke",
                            name=f"lamp-no-recovery-{RUN_DIR.name}", resume="never",
                            dir=str(RUN_DIR), config=configuration())
    if LOCAL_CHECKPOINT_PATH is not None:
        path = select_checkpoint(LOCAL_CHECKPOINT_PATH, CHECKPOINT_SELECTOR)
        return path, checkpoint_provenance(path, WANDB_ARTIFACT, expected_hash=LOCAL_EXPECTED_SHA256)
    artifact = WB_RUN.use_artifact(WANDB_ARTIFACT, type="model")
    root = artifact.download(root=str(RUN_DIR / "source_artifact"))
    path = select_checkpoint(root, CHECKPOINT_SELECTOR)
    provenance = checkpoint_provenance(path, WANDB_ARTIFACT, resolved_artifact=artifact.qualified_name)
    provenance.update({"artifact_digest": artifact.digest, "artifact_id": artifact.id,
                       "artifact_version": artifact.version, "artifact_relative_path": path.relative_to(Path(root).resolve()).as_posix()})
    return path, provenance

In [ ]:
with checked_stage("checkpoint_acquisition"):
    BASELINE_PATH, STATE["provenance"] = obtain_checkpoint()
    print(json.dumps(STATE["provenance"], indent=2))

## 05 — Memuat dan Menganalisis YOLO26n
Audit menggunakan checkpoint sebenarnya, bukan asumsi enam kelas. Backbone/neck diturunkan
dari YAML dan sumber fitur head. Model harus YOLO26n deteksi dengan kedua cabang yang utuh.
Ringkasan tahap ditampilkan; daftar Conv/BN, residual, attention, concat, dan skip disimpan ke JSON.

In [ ]:
def ordered_names(names):
    """Pertahankan identitas dan urutan indeks kelas."""
    if isinstance(names, dict):
        indexed = {int(k): str(v) for k, v in names.items()}
        if sorted(indexed) != list(range(len(indexed))):
            raise ValueError("Indeks kelas wajib berurutan mulai dari nol.")
        return [indexed[i] for i in range(len(indexed))]
    return [str(value) for value in names]


def assert_class_match(model_names, dataset_names):
    """Jangan remap kelas diam-diam untuk membuat evaluasi berjalan."""
    if ordered_names(model_names) != ordered_names(dataset_names):
        raise ValueError(f"Urutan/nama kelas berbeda: model={ordered_names(model_names)}, dataset={ordered_names(dataset_names)}")


def architecture_audit(core):
    """Bangun allowlist hidden channel berdasarkan tipe modul dan hubungan konsumen."""
    if not isinstance(core, DetectionModel):
        raise TypeError("Checkpoint bukan DetectionModel Ultralytics.")
    heads = [(name, module) for name, module in core.named_modules() if isinstance(module, Detect)]
    if len(heads) != 1:
        raise ValueError("Diperlukan tepat satu Detect head.")
    head_name, head = heads[0]
    cfg = core.yaml
    first_conv = next(module for module in core.modules() if isinstance(module, torch.nn.Conv2d))
    if "yolo26" not in str(cfg.get("yaml_file", "")).lower() or cfg.get("reg_max") != 1 or first_conv.out_channels != 16:
        raise ValueError("Metadata/struktur tidak cocok dengan YOLO26n standar; audit manual diperlukan.")
    if cfg.get("scale") not in (None, "", "n"):
        raise ValueError("Checkpoint bukan skala nano.")
    branches = ("cv2", "cv3", "one2one_cv2", "one2one_cv3")
    if any(getattr(head, attr, None) is None or len(getattr(head, attr)) != head.nl for attr in branches):
        raise ValueError("Kedua cabang head wajib tersedia; checkpoint fused/head berubah ditolak.")
    names = ordered_names(core.names)
    if len(names) != head.nc or head.no != head.nc + 4 * head.reg_max:
        raise ValueError("Metadata kelas/dimensi Detect tidak konsisten.")
    modules = dict(core.named_modules())
    attention_prefixes = [name for name, m in modules.items() if isinstance(m, (C2PSA, Attention, AAttn))]
    allowed = {}
    for name, module in modules.items():
        if not isinstance(module, Bottleneck) or name.startswith(head_name + "."):
            continue
        if any(name == prefix or name.startswith(prefix + ".") for prefix in attention_prefixes):
            continue
        producer, consumer = module.cv1.conv, module.cv2.conv
        bn = getattr(module.cv1, "bn", None)
        if not isinstance(bn, torch.nn.BatchNorm2d) or producer.groups != 1 or consumer.groups != 1:
            raise ValueError(f"Bottleneck {name} tidak mendukung kebijakan hidden-channel konservatif.")
        if producer.out_channels != consumer.in_channels or producer.out_channels != bn.num_features:
            raise ValueError(f"Dimensi hidden channel {name} tidak konsisten.")
        allowed[name + ".cv1.conv"] = {"root": producer, "bn": bn, "consumer": consumer,
                                        "initial_channels": producer.out_channels, "block": name}
    if not allowed:
        raise ValueError("Tidak ditemukan hidden-channel group yang eligible.")
    backbone_count = len(cfg["backbone"])
    stages = []
    for i, module in enumerate(core.model):
        stages.append({"index": i, "type": type(module).__name__, "from": module.f,
                       "region": "detection_head" if module is head else ("backbone" if i < backbone_count else "neck"),
                       "parameters": sum(p.numel() for p in module.parameters())})
    inventory = [{"name": name, "type": type(module).__name__,
                  "residual_add": bool(getattr(module, "add", False))}
                 for name, module in modules.items()
                 if isinstance(module, (torch.nn.Conv2d, torch.nn.BatchNorm2d, Bottleneck, C2PSA, Attention, AAttn))
                 or type(module).__name__ in {"Concat", "C3k2", "SPPF"}]
    protection = []
    for name, module in modules.items():
        if isinstance(module, torch.nn.Conv2d) and name not in allowed:
            reason = "head one2many/one2one utuh" if name.startswith(head_name + ".") else "dimensi CSP/residual/attention/antartahap dilindungi"
            protection.append({"name": name, "reason": reason,
                               "protected_dimension": "all" if name.startswith(head_name + ".") else "out_channels",
                               "dependent_input_pruning_allowed": any(module is entry["consumer"] for entry in allowed.values())})
    return {"head": head, "head_name": head_name, "eligible": allowed, "modules": modules,
            "report": {"stages": stages, "module_inventory": inventory, "protected_layers": protection,
                       "eligible_layers": list(allowed), "head_feature_sources": list(head.f),
                       "class_names": names, "nc": head.nc, "reg_max": head.reg_max, "nl": head.nl,
                       "end2end": head.end2end, "unknown_policy": "fail explicitly"}}


def raw_branches(output):
    """Ambil raw kedua cabang tanpa mengganti forward atau algoritme deteksi."""
    if not isinstance(output, tuple) or len(output) != 2 or not isinstance(output[1], dict):
        raise ValueError("Output eval YOLO26 harus (decoded, raw_predictions).")
    branches = output[1]
    if set(branches) != {"one2many", "one2one"}:
        raise ValueError(f"Cabang YOLO26 tidak lengkap: {list(branches)}")
    return branches


def trace_outputs(output):
    """Root autograd mencakup boxes/scores kedua cabang, tanpa top-k/NMS."""
    branches = raw_branches(output)
    return [branches[branch][key] for branch in ("one2many", "one2one") for key in ("boxes", "scores")]


def finite_tensors(value):
    """Validasi semua tensor, termasuk feature maps dalam output nested."""
    if isinstance(value, torch.Tensor):
        if not torch.isfinite(value).all():
            raise ValueError("Output model mengandung NaN/Inf.")
    elif isinstance(value, dict):
        for item in value.values():
            finite_tensors(item)
    elif isinstance(value, (tuple, list)):
        for item in value:
            finite_tensors(item)


def forward_snapshot(core, image):
    """Forward FP32, cek kontrak raw/decoded, simpan tensor kecil untuk restore parity."""
    core.eval()
    with torch.no_grad():
        output = core(image)
    finite_tensors(output)
    branches = raw_branches(output)
    head = core.model[-1]
    snapshot = {"decoded": output[0].detach().cpu().clone()}
    anchors = None
    for branch in ("one2many", "one2one"):
        boxes, scores = branches[branch]["boxes"], branches[branch]["scores"]
        if boxes.ndim != 3 or scores.ndim != 3 or boxes.shape[:2] != (image.shape[0], 4 * head.reg_max):
            raise ValueError(f"Bentuk boxes {branch} invalid: {tuple(boxes.shape)}")
        if scores.shape[:2] != (image.shape[0], head.nc) or boxes.shape[2] != scores.shape[2]:
            raise ValueError(f"Bentuk scores {branch} tidak cocok dengan jumlah kelas.")
        if anchors is not None and anchors != boxes.shape[2]:
            raise ValueError("Jumlah anchor kedua cabang berbeda.")
        anchors = boxes.shape[2]
        snapshot[branch + "/boxes"] = boxes.detach().cpu().clone()
        snapshot[branch + "/scores"] = scores.detach().cpu().clone()
    decoded = snapshot["decoded"]
    if head.end2end:
        if decoded.ndim != 3 or decoded.shape[0] != image.shape[0] or decoded.shape[-1] != 6:
            raise ValueError("Output end-to-end bukan [batch, detections, 6].")
        ids = decoded[..., 5]
        if ((ids < 0) | (ids >= head.nc) | (ids != ids.floor())).any():
            raise ValueError("Indeks kelas pada output deteksi invalid.")
    elif decoded.ndim != 3 or decoded.shape[:2] != (image.shape[0], head.nc + 4):
        raise ValueError("Output one-to-many tidak memiliki dimensi nc+4.")
    return snapshot


def model_measurements(core, imgsz):
    """Hitung Conv/BN dan attention matmul tanpa menghilangkan child Conv attention."""
    parameters = sum(p.numel() for p in core.parameters())
    copy = deepcopy(core).eval()
    parameter = next(copy.parameters())
    sample = torch.zeros(1, 3, imgsz, imgsz, device=parameter.device, dtype=torch.float32)
    attention_macs = []
    def count_attention(module, inputs, output):
        counter = SimpleNamespace(total_ops=0, num_heads=module.num_heads, head_dim=module.head_dim,
                                  key_dim=getattr(module, "key_dim", module.head_dim), area=getattr(module, "area", 1))
        _attention_ops(counter, inputs, output)
        attention_macs.append(float(counter.total_ops))
    handles = [module.register_forward_hook(count_attention) for module in copy.modules()
               if isinstance(module, (Attention, AAttn))]
    try:
        with torch.no_grad():
            # A custom Attention handler in classic THOP would hide its child Conv/BN counts.
            macs, _ = thop_profile(copy, inputs=(sample,), verbose=False)
    finally:
        for handle in handles:
            handle.remove()
    gflops = (float(macs) + sum(attention_macs)) * 2 / 1e9
    if not math.isfinite(gflops) or gflops <= 0:
        raise RuntimeError("Profiling GFLOPs gagal/bernilai nol; cek thop dan source dependency.")
    return {"parameters": parameters, "gflops": gflops, "precision": "FP32", "fusion": False}


def head_snapshot(head):
    """Salin parameter/buffer serta kontrak dimensi seluruh head yang dilindungi."""
    return {"state": {key: value.detach().cpu().clone() for key, value in head.state_dict().items()},
            "contract": (head.nc, head.no, head.nl, head.reg_max, head.end2end)}


def assert_head_unchanged(head, before):
    """Pruning tersembunyi tidak boleh menyentuh bobot atau dimensi head."""
    after = head_snapshot(head)
    if after["contract"] != before["contract"] or after["state"].keys() != before["state"].keys():
        raise ValueError("Kontrak/struktur detection head berubah.")
    if any(not torch.equal(after["state"][key], value) for key, value in before["state"].items()):
        raise ValueError("Parameter/buffer detection head berubah.")

In [ ]:
with checked_stage("baseline_load_and_audit"):
    BASELINE = YOLO(str(BASELINE_PATH), task="detect")
    BASELINE_CORE = BASELINE.model.to(TARGET_DEVICE).float().eval()
    AUDIT = architecture_audit(BASELINE_CORE)
    STATE["checks"]["baseline_loaded"] = True
    safe_keys = ("epoch", "best_fitness", "date", "version", "license")
    STATE["provenance"]["checkpoint_metadata"] = {key: BASELINE.ckpt.get(key) for key in safe_keys
                                                   if isinstance(BASELINE.ckpt.get(key), (str, int, float, bool, type(None)))}
    STATE["provenance"]["model_class"] = type(BASELINE_CORE).__name__
    source_args = BASELINE.ckpt.get("train_args", {})
    STATE["provenance"]["source_training_args"] = {
        key: source_args.get(key) for key in ("task", "model", "data", "seed", "epochs", "imgsz", "batch", "nms", "pretrained")
        if isinstance(source_args.get(key), (str, int, float, bool, type(None)))
    }
    STATE["provenance"]["class_names"] = ordered_names(BASELINE_CORE.names)
    STATE["provenance"]["yaml"] = deepcopy(BASELINE_CORE.yaml)
    IMAGE = torch.rand(1, 3, IMGSZ, IMGSZ, device=TARGET_DEVICE, dtype=torch.float32)
    BASELINE_OUTPUT = forward_snapshot(BASELINE_CORE, IMAGE)
    STATE["checks"]["baseline_forward"] = True
    BEFORE = model_measurements(BASELINE_CORE, IMGSZ)
    HEAD_BEFORE = head_snapshot(AUDIT["head"])
    STATE["metrics"]["baseline"] = BEFORE
    write_json(RUN_DIR / "architecture_audit.json", AUDIT["report"])
    print("Classes:", ordered_names(BASELINE_CORE.names), "|", BEFORE)
    display(pd.DataFrame(AUDIT["report"]["stages"]))

## 06 — Analisis Dependency Graph
Allowlist membatasi root ke hidden Conv Bottleneck. Setiap group hanya boleh mengubah
output Conv tersebut, BatchNorm terkait, dan input Conv berikutnya. CSP `chunk/split`,
residual output, attention, serta seluruh head dilindungi. Tidak ada rewrite C2f/C3k2.
Graph ditelusuri pada `eval()` sehingga one-to-one tidak terputus oleh `detach()` training.

In [ ]:
def layer_dimensions(core):
    """Catat dimensi aktual Conv/BN untuk laporan struktural sebelum/sesudah."""
    rows = {}
    for name, module in core.named_modules():
        if isinstance(module, torch.nn.Conv2d):
            rows[name] = {"type": "Conv2d", "in_channels": module.in_channels,
                          "out_channels": module.out_channels, "groups": module.groups,
                          "weight_shape": list(module.weight.shape)}
        elif isinstance(module, torch.nn.BatchNorm2d):
            rows[name] = {"type": "BatchNorm2d", "in_channels": module.num_features,
                          "out_channels": module.num_features, "groups": None,
                          "weight_shape": list(module.weight.shape)}
    return rows


def audit_group(group, graph, audit):
    """Validasi batas perubahan setiap dependency group sebelum pruning fisik."""
    root = group[0][0].target.module
    entries = [(name, entry) for name, entry in audit["eligible"].items() if entry["root"] is root]
    if len(entries) != 1:
        raise ValueError("Dependency group berakar di luar allowlist.")
    name, entry = entries[0]
    for dependency, _indices in group:
        module, handler = dependency.target.module, dependency.handler
        if isinstance(module, torch.nn.Conv2d):
            valid = (module is entry["root"] and graph.is_out_channel_pruning_fn(handler)) or (
                module is entry["consumer"] and graph.is_in_channel_pruning_fn(handler))
            if not valid:
                raise ValueError(f"Group {name} menjangkau Conv/dimensi terlindungi: {dependency}")
        elif isinstance(module, torch.nn.BatchNorm2d):
            if module is not entry["bn"]:
                raise ValueError(f"Group {name} menjangkau BN terlindungi.")
        elif isinstance(module, torch.nn.Module) and any(True for _ in module.parameters(recurse=False)):
            raise ValueError(f"Modul berparameter belum didukung dalam group {name}: {type(module).__name__}")
    return name, entry


def build_pruner(core, image, audit):
    """Bangun DepGraph dengan importance LAMP bawaan dan audit seluruh group eligible."""
    core.eval().requires_grad_(True)
    eligible_roots = {entry["root"] for entry in audit["eligible"].values()}
    ignored = [module for module in core.modules() if isinstance(module, torch.nn.Conv2d) and module not in eligible_roots]
    ignored.append(audit["head"])
    importance = tp.importance.LAMPImportance(p=IMPORTANCE_P, group_reduction="mean", bias=False)
    with torch.enable_grad(), warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter("always")
        pruner = tp.pruner.BasePruner(
            core, image, importance=importance, global_pruning=GLOBAL_PRUNING,
            pruning_ratio=PRUNING_RATIO, iterative_steps=ITERATIVE_STEPS,
            max_pruning_ratio=MAX_PRUNING_RATIO, round_to=ROUND_TO,
            ignored_layers=ignored, root_module_types=[torch.nn.Conv2d],
            forward_fn=lambda model, example: model(example), output_transform=trace_outputs,
        )
    diagnostics = [str(item.message) for item in captured]
    for message in diagnostics:
        print("Dependency warning:", message)
    groups = list(pruner.DG.get_all_groups(ignored_layers=ignored, root_module_types=[torch.nn.Conv2d]))
    names = []
    for group in groups:
        name, entry = audit_group(group, pruner.DG, audit)
        score = importance(group)
        if score is None or score.numel() != entry["root"].out_channels or not torch.isfinite(score).all():
            raise ValueError(f"Importance LAMP invalid untuk {name}.")
        names.append(name)
    if set(names) != set(audit["eligible"]):
        raise ValueError(f"DepGraph tidak mencakup semua kandidat: {sorted(set(audit['eligible']) - set(names))}")
    if any(module.training for module in core.modules()):
        raise ValueError("Graph harus dibangun dalam mode eval seluruh modul.")
    return pruner, {"eligible_groups": sorted(names), "group_count": len(groups), "warnings": diagnostics}

In [ ]:
with checked_stage("dependency_graph"):
    PRUNED_CORE = deepcopy(BASELINE_CORE)
    PRUNED_AUDIT = architecture_audit(PRUNED_CORE)
    DIMS_BEFORE = layer_dimensions(PRUNED_CORE)
    PRUNER, GRAPH_REPORT = build_pruner(PRUNED_CORE, IMAGE, PRUNED_AUDIT)
    assert_head_unchanged(PRUNED_AUDIT["head"], HEAD_BEFORE)
    STATE["checks"]["dependency_graph"] = True
    STATE["manifest"].update({"graph": GRAPH_REPORT, "eligible_layers": list(PRUNED_AUDIT["eligible"]),
                              "protected_layers": PRUNED_AUDIT["report"]["protected_layers"]})
    STATE["warnings"].extend(GRAPH_REPORT["warnings"])
    print("Group eligible:", GRAPH_REPORT["group_count"])
    display(pd.DataFrame([{"layer": name, "hidden_channels": entry["initial_channels"]}
                          for name, entry in PRUNED_AUDIT["eligible"].items()]))

## 07 — Implementasi LAMP Structured Pruning
`BasePruner.step(interactive=True)` memilih indeks melalui ranking LAMP bawaan.
Indeks/group diaudit sebelum `group.prune()`. Tidak ada masking ataupun regularisasi sparsity.
Cap 50% juga diperiksa eksplisit pada tiap root, bukan hanya dipercayakan ke argumen library.

In [ ]:
def apply_structural_pruning(pruner, audit):
    """Prune satu langkah; hentikan sebelum mutasi jika group melanggar batas."""
    history = []
    for group in pruner.step(interactive=True):
        name, entry = audit_group(group, pruner.DG, audit)
        if not pruner.DG.check_pruning_group(group):
            raise ValueError(f"Dependency group tidak valid: {name}")
        indices = [int(index) for index in group[0][1]]
        if not indices:
            continue  # BasePruner dapat mengeluarkan group dengan nol indeks terpilih.
        remaining = entry["root"].out_channels - len(indices)
        removed_total = entry["initial_channels"] - remaining
        if remaining < 1 or removed_total / entry["initial_channels"] > MAX_PRUNING_RATIO + 1e-12:
            raise ValueError(f"Group {name} melebihi cap atau menghapus semua channel.")
        history.append({"root": name, "removed_indices": indices,
                        "channels_before": entry["root"].out_channels, "channels_after": remaining,
                        "dependencies": [str(dependency) for dependency, _ in group]})
        group.prune()
    if not history:
        raise RuntimeError("Tidak ada group yang dipangkas; rasio/rounding/constraint mungkin terlalu ketat.")
    return history


def reduction(before, after):
    """Rasio aktual dengan denominator eksplisit dan tervalidasi."""
    if before <= 0 or after < 0 or after > before:
        raise ValueError(f"Reduksi invalid: {before} -> {after}")
    return (before - after) / before


def write_layer_report(path, before, after, audit):
    """Laporkan setiap Conv/BN, termasuk consumer dan layer yang dilindungi."""
    fields = ["layer", "type", "in_before", "in_after", "out_before", "out_after", "weight_before", "weight_after", "policy", "changed"]
    roots = {entry["root"] for entry in audit["eligible"].values()}
    permitted = roots | {entry["bn"] for entry in audit["eligible"].values()} | {entry["consumer"] for entry in audit["eligible"].values()}
    modules = audit["modules"]
    if set(before) != set(after):
        raise ValueError("Pruning menghapus/mengganti modul secara tak terduga.")
    with Path(path).open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        for name, old in before.items():
            new = after[name]
            changed = old != new
            if changed and modules[name] not in permitted:
                raise ValueError(f"Dimensi layer terlindungi berubah: {name}")
            writer.writerow({"layer": name, "type": old["type"], "in_before": old["in_channels"], "in_after": new["in_channels"],
                             "out_before": old["out_channels"], "out_after": new["out_channels"],
                             "weight_before": old["weight_shape"], "weight_after": new["weight_shape"],
                             "policy": "eligible root" if name in audit["eligible"] else ("dependent BN/consumer" if modules[name] in permitted else "protected"),
                             "changed": changed})

In [ ]:
with checked_stage("structural_pruning"):
    PRUNING_HISTORY = apply_structural_pruning(PRUNER, PRUNED_AUDIT)
    PRUNED_CORE.requires_grad_(False).eval()
    DIMS_AFTER = layer_dimensions(PRUNED_CORE)
    write_layer_report(RUN_DIR / "layer_pruning_report.csv", DIMS_BEFORE, DIMS_AFTER, PRUNED_AUDIT)
    initial_channels = sum(entry["initial_channels"] for entry in PRUNED_AUDIT["eligible"].values())
    remaining_channels = sum(entry["root"].out_channels for entry in PRUNED_AUDIT["eligible"].values())
    actual_ratio = reduction(initial_channels, remaining_channels)
    STATE["checks"]["structural_channels_removed"] = actual_ratio > 0
    STATE["manifest"].update({"pruned_groups": PRUNING_HISTORY, "channels_before": initial_channels,
                              "channels_after": remaining_channels, "actual_eligible_channel_ratio": actual_ratio,
                              "ratio_denominator": "jumlah output channel root eligible unik; tidak termasuk protected layers",
                              "layer_dimensions_before": DIMS_BEFORE, "layer_dimensions_after": DIMS_AFTER})
    if abs(actual_ratio - PRUNING_RATIO) > 1e-12:
        STATE["warnings"].append("Rasio aktual berbeda dari target karena integer channel, threshold ranking/cap, atau ROUND_TO; lihat group history.")
    print(f"Hidden channel eligible: {initial_channels} -> {remaining_channels}; actual={actual_ratio:.4%}")

## 08 — Validasi Struktur Hasil Pruning
Periksa dimensi Conv/BN, forward kedua cabang, tensor finite, dan head yang tetap identik.
Parameter harus berkurang; GFLOPs diukur ulang pada salinan model unfused dengan metode yang sama.
Profiling memakai input resolusi penuh: THOP menghitung Conv/BN, sedangkan hook Ultralytics
menambahkan matmul attention secara terpisah agar child Conv attention tetap terhitung.

In [ ]:
def assert_layer_consistency(core):
    """Dimensi atribut modul harus sesuai dengan tensor parameter/buffer aktual."""
    for name, module in core.named_modules():
        if isinstance(module, torch.nn.Conv2d):
            expected = (module.out_channels, module.in_channels // module.groups, *module.kernel_size)
            if tuple(module.weight.shape) != expected or module.in_channels % module.groups or module.out_channels % module.groups:
                raise ValueError(f"Bentuk Conv invalid pada {name}.")
        elif isinstance(module, torch.nn.BatchNorm2d):
            if any(t.numel() != module.num_features for t in (module.weight, module.bias, module.running_mean, module.running_var)):
                raise ValueError(f"Bentuk BatchNorm invalid pada {name}.")

In [ ]:
with checked_stage("pruned_structure_and_forward"):
    assert_layer_consistency(PRUNED_CORE)
    PRUNED_OUTPUT = forward_snapshot(PRUNED_CORE, IMAGE)
    STATE["checks"]["pruned_forward"] = True
    assert_head_unchanged(PRUNED_AUDIT["head"], HEAD_BEFORE)
    STATE["checks"]["protected_head_unchanged"] = True
    AFTER = model_measurements(PRUNED_CORE, IMGSZ)
    if AFTER["parameters"] >= BEFORE["parameters"]:
        raise RuntimeError("Jumlah parameter tidak berkurang; structural smoke test gagal.")
    STATE["checks"]["parameters_reduced"] = True
    STATE["metrics"]["pruned"] = AFTER
    STATE["manifest"].update({"parameters_before": BEFORE["parameters"], "parameters_after": AFTER["parameters"],
                              "gflops_before": BEFORE["gflops"], "gflops_after": AFTER["gflops"],
                              "parameter_reduction_ratio": reduction(BEFORE["parameters"], AFTER["parameters"]),
                              "gflops_reduction_ratio": reduction(BEFORE["gflops"], AFTER["gflops"])})
    print("Sebelum:", BEFORE, "\nSesudah:", AFTER)

## 09 — Menyimpan dan Memuat Ulang Model
Structural pruning mengubah bentuk tensor: state dict tidak dimuat ke arsitektur YOLO26n standar.
Checkpoint menyimpan objek `DetectionModel` asli yang telah dipangkas, seluruhnya FP32,
`ema=None`, dan `optimizer=None`. Tidak ada kelas custom notebook yang perlu diimpor saat restore.
`YOLO.save()` tidak dipakai karena versi ini menyalin model ke FP16.

Uji wajib memakai proses Python baru tanpa `PYTHONPATH` repository: load lewat `YOLO(path)`,
forward, dan `predict()`. Toleransi raw/decoded FP32 adalah `rtol=1e-4`, `atol=1e-5`.
Muat hanya checkpoint dari sumber yang dipercaya; format ini menyerialisasi objek model Python.

In [ ]:
RELOAD_SCRIPT = r'''
import json, sys
from pathlib import Path
import numpy as np
import torch
from ultralytics import YOLO

checkpoint, reference, result, device = sys.argv[1:]
torch.set_num_threads(2)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision("highest")
payload = torch.load(reference, map_location="cpu", weights_only=True)
wrapper = YOLO(checkpoint, task="detect")
core = wrapper.model.to(device).float().eval()
with torch.no_grad():
    output = core(payload["image"].to(device))
snapshot = {"decoded": output[0].detach().cpu()}
for branch in ("one2many", "one2one"):
    for key in ("boxes", "scores"):
        snapshot[branch + "/" + key] = output[1][branch][key].detach().cpu()
for key, expected in payload["expected"].items():
    if not torch.isfinite(snapshot[key]).all():
        raise RuntimeError("Non-finite output after reload: " + key)
    torch.testing.assert_close(snapshot[key], expected, rtol=1e-4, atol=1e-5)
if sum(p.numel() for p in core.parameters()) != payload["parameters"]:
    raise RuntimeError("Parameter count changed after reload")
if not all(p.dtype == torch.float32 for p in core.parameters()):
    raise RuntimeError("Reload is not FP32")
torch.save(snapshot, result)
# Fresh wrapper protects the raw unfused model from inference backend fusion.
predictor = YOLO(checkpoint, task="detect")
size = payload["image"].shape[-1]
predictions = predictor.predict(source=np.zeros((size, size, 3), dtype=np.uint8),
    imgsz=size, device=device, conf=0.001, iou=0.7, max_det=300,
    quantize=None, nms=None, verbose=False, save=False)
if len(predictions) != 1 or predictions[0].boxes is None:
    raise RuntimeError("No valid detection Results after reload")
boxes = predictions[0].boxes.data.detach().cpu()
if boxes.ndim != 2 or boxes.shape[1] != 6 or not torch.isfinite(boxes).all():
    raise RuntimeError("Invalid detection tensor")
nc = len(core.names)
if boxes.numel() and (((boxes[:, 5] < 0) | (boxes[:, 5] >= nc) | (boxes[:, 5] != boxes[:, 5].floor())).any()
                     or ((boxes[:, 4] < 0) | (boxes[:, 4] > 1)).any()):
    raise RuntimeError("Invalid detection class/confidence")
Path(result).with_suffix(".json").write_text(json.dumps({"prediction_count": len(boxes),
    "parameters": payload["parameters"], "direct_yolo_load": True,
    "raw_parity": True, "prediction_shape": list(boxes.shape)}), encoding="utf-8")
'''


def save_fp32_checkpoint(core, path, provenance):
    """Simpan arsitektur hasil pruning memakai format model-object Ultralytics."""
    model = deepcopy(core).cpu().float().eval().requires_grad_(False)
    for module in model.modules():
        module._forward_hooks.clear()
        module._forward_pre_hooks.clear()
        module._backward_hooks.clear()
    model.args = {"task": "detect", "imgsz": IMGSZ, "data": str(DATA_YAML), "nms": None, "quantize": None}
    torch.save({"model": model, "ema": None, "optimizer": None, "epoch": -1,
                "train_args": dict(model.args), "version": ultralytics.__version__,
                "date": datetime.now(timezone.utc).isoformat(), "lamp_format_version": 1,
                "lamp_provenance": provenance, "recovery_enabled": False}, path)
    return {"path": str(Path(path).resolve()), "sha256": sha256_file(path), "file_bytes": Path(path).stat().st_size}


def verify_fresh_reload(checkpoint, core, image, output_dir, *, timeout=300):
    """Restore di interpreter baru; exit code/error tidak boleh disembunyikan."""
    output_dir = Path(output_dir).resolve()
    script = output_dir / "verify_reload.py"
    reference = output_dir / "reload_reference.pt"
    result = output_dir / "reload_output.pt"
    script.write_text(RELOAD_SCRIPT, encoding="utf-8")
    expected = forward_snapshot(core, image)
    torch.save({"image": image.detach().cpu(), "expected": expected,
                "parameters": sum(p.numel() for p in core.parameters())}, reference)
    child_env = dict(os.environ)
    child_env.pop("PYTHONPATH", None)
    child_env["WANDB_MODE"] = "disabled"
    child_env["YOLO_AUTOINSTALL"] = "false"
    completed = subprocess.run([sys.executable, "-B", str(script), str(Path(checkpoint).resolve()),
                                str(reference), str(result), str(next(core.parameters()).device)],
                               cwd=output_dir, env=child_env, capture_output=True, text=True,
                               encoding="utf-8", errors="replace", timeout=timeout)
    (output_dir / "reload_stdout.txt").write_text(completed.stdout, encoding="utf-8")
    (output_dir / "reload_stderr.txt").write_text(completed.stderr, encoding="utf-8")
    if completed.returncode:
        raise RuntimeError(f"Fresh-process reload gagal (exit {completed.returncode}): {completed.stderr[-6000:]}")
    actual = torch.load(result, map_location="cpu", weights_only=True)
    for key in expected:
        torch.testing.assert_close(actual[key], expected[key], rtol=1e-4, atol=1e-5)
    report = json.loads(result.with_suffix(".json").read_text(encoding="utf-8"))
    report["max_absolute_difference"] = {key: float((actual[key] - expected[key]).abs().max()) for key in expected}
    report["process"] = "fresh Python, no repository PYTHONPATH"
    return report

In [ ]:
with checked_stage("save_and_fresh_reload"):
    PRUNED_PATH = RUN_DIR / "pruned_fp32.pt"
    BASELINE_COMPARISON_PATH = RUN_DIR / "baseline_fp32.pt"
    baseline_file = save_fp32_checkpoint(BASELINE_CORE, BASELINE_COMPARISON_PATH, STATE["provenance"])
    pruned_file = save_fp32_checkpoint(PRUNED_CORE, PRUNED_PATH, STATE["provenance"])
    STATE["checks"]["checkpoint_saved"] = True
    RELOAD_REPORT = verify_fresh_reload(PRUNED_PATH, PRUNED_CORE, IMAGE, RUN_DIR, timeout=RELOAD_TIMEOUT_SECONDS)
    RELOADED = YOLO(str(PRUNED_PATH), task="detect").model.to(TARGET_DEVICE).float().eval()
    assert_layer_consistency(RELOADED)
    assert_head_unchanged(RELOADED.model[-1], HEAD_BEFORE)
    forward_snapshot(RELOADED, IMAGE)
    for key in ("checkpoint_reloaded", "reload_consistent", "detection_prediction"):
        STATE["checks"][key] = True
    BEFORE["checkpoint_bytes"] = baseline_file["file_bytes"]
    AFTER["checkpoint_bytes"] = pruned_file["file_bytes"]
    STATE["manifest"].update({"saved_baseline": baseline_file, "saved_pruned": pruned_file,
                              "serialization": "native Ultralytics model object, FP32, ema=None, optimizer=None",
                              "reload": RELOAD_REPORT, "save_reload_status": "PASS", "inference_status": "PASS"})
    if not all(STATE["checks"].values()):
        raise RuntimeError(f"Syarat smoke test belum terpenuhi: {STATE['checks']}")
    STATE["structural_smoke_status"] = "PASS"
    restore = (
        "# Restore checkpoint LAMP\n\n"
        "Gunakan Python >=3.11 serta torch/torchvision yang kompatibel dengan runtime asal.\n"
        f"Versi runtime asal: {json.dumps(STATE['environment']['packages'])}\n\n"
        "Checkpoint menyimpan objek DetectionModel dengan bentuk tensor yang telah berubah. "
        "Jangan muat state dict ke yolo26n.yaml standar. Muat hanya berkas yang dipercaya.\n\n"
        "```python\nfrom ultralytics import YOLO\n"
        "model = YOLO('pruned_fp32.pt', task='detect')\n"
        "results = model.predict('image.jpg', imgsz=640, nms=None, quantize=None)\n```\n\n"
        "verify_reload.py memeriksa forward parity dan predict pada proses baru; "
        "lihat reload_reference.pt, reload_output.pt, dan pruning_manifest.json.\n"
        "Arsitektur tidak memerlukan kelas custom atau import repository. "
        "Tidak ada recovery/training yang dijalankan.\n"
    )
    (RUN_DIR / "RESTORE.md").write_text(restore, encoding="utf-8")
    print("Fresh reload:", RELOAD_REPORT)

## 10 — Evaluasi pada Dataset Asli
Default mengikuti notebook baseline Poltrada V3: **val**, 640, confidence 0.001, IoU 0.7,
max_det 300, FP32, `nms=None`. Pada versi ini `nms=None` memakai jalur default evaluator;
jangan menyamakannya dengan `nms=False` yang memilih head NMS-free.
Baseline dan hasil reload memakai dataset dan pengaturan identik. Tidak ada tuning/test-set selection.
Dataset tidak tersedia → `NOT_RUN`; dataset ada tetapi kelas/path invalid → `FAIL` dan exception.
Salinan YAML evaluasi menghapus instruksi download; dataset asli tidak diubah.

In [ ]:
def prepare_validation_dataset(data_yaml, dataset_root, output_dir):
    """Validasi dataset lokal tanpa download atau normalisasi label otomatis."""
    source = Path(data_yaml).expanduser()
    if not source.is_file():
        return None
    raw = yaml.safe_load(source.read_text(encoding="utf-8"))
    if not isinstance(raw, dict) or not raw.get("val") or not raw.get("names"):
        raise ValueError("DATA_YAML tersedia tetapi tidak berisi val/names yang valid.")
    raw.pop("download", None)
    root = Path(dataset_root).expanduser().resolve() if dataset_root is not None else source.resolve().parent
    if not root.is_dir():
        raise FileNotFoundError(f"DATASET_ROOT tidak tersedia: {root}")
    raw["path"] = str(root)
    copy_path = Path(output_dir) / "evaluation_data.yaml"
    copy_path.write_text(yaml.safe_dump(raw, sort_keys=False, allow_unicode=True), encoding="utf-8")
    info = check_det_dataset(str(copy_path), autodownload=False)
    splits = info["val"] if isinstance(info["val"], list) else [info["val"]]
    if any(not Path(split).exists() for split in splits):
        raise FileNotFoundError("Validation split tidak tersedia.")
    return {"yaml": str(copy_path.resolve()), "info": info, "source_yaml": str(source.resolve()),
            "source_yaml_sha256": sha256_file(source), "evaluation_yaml_sha256": sha256_file(copy_path),
            "val_paths": [str(split) for split in splits], "class_names": ordered_names(info["names"])}


def validation_arguments(data_yaml, output_dir, label):
    """Satu sumber pengaturan evaluasi untuk baseline dan pruned model."""
    return {"data": str(data_yaml), "split": "val", "imgsz": IMGSZ, "batch": BATCH,
            "workers": WORKERS, "device": str(TARGET_DEVICE), "conf": 0.001,
            "iou": 0.7, "max_det": 300, "quantize": None, "nms": None,
            "augment": False, "plots": False, "verbose": False, "save_json": False,
            "project": str(output_dir), "name": label, "exist_ok": False}


def evaluate_checkpoint(path, dataset, output_dir, label):
    """Evaluasi wrapper baru agar fusion validator tidak mengubah model pengukuran."""
    wrapper = YOLO(str(path), task="detect")
    assert_class_match(wrapper.names, dataset["info"]["names"])
    metric = wrapper.val(**validation_arguments(dataset["yaml"], output_dir, label))
    box = metric.box
    per_class = [{"class_id": i, "name": name, "ap50": None, "ap50_95": None}
                 for i, name in enumerate(dataset["class_names"])]
    for row, class_id in enumerate(box.ap_class_index):
        per_class[int(class_id)].update({"ap50": float(box.ap50[row]), "ap50_95": float(box.ap[row])})
    result = {"map50_95": float(box.map), "map50": float(box.map50),
              "precision": float(box.mp), "recall": float(box.mr), "per_class": per_class,
              "split": "val", "precision_mode": "FP32", "nms": None}
    if any(not math.isfinite(result[key]) for key in ("map50_95", "map50", "precision", "recall")):
        raise ValueError("Metrik evaluasi mengandung NaN/Inf.")
    return result


def cuda_latency(core, image, warmup=50, repeats=200):
    """Latency forward model batch satu; tidak mencakup preprocessing/NMS/disk IO."""
    if image.device.type != "cuda":
        return {"status": "NOT_RUN", "reason": "CUDA tidak tersedia; tidak mengklaim benchmark GPU."}
    core = deepcopy(core).eval().requires_grad_(False)
    samples = []
    with torch.inference_mode():
        for _ in range(warmup):
            core(image)
        torch.cuda.synchronize(image.device)
        for _ in range(repeats):
            torch.cuda.synchronize(image.device)
            started = time.perf_counter()
            core(image)
            torch.cuda.synchronize(image.device)
            samples.append((time.perf_counter() - started) * 1000)
    return {"status": "PASS", "median_ms": float(np.median(samples)), "mean_ms": float(np.mean(samples)),
            "p05_ms": float(np.percentile(samples, 5)), "p95_ms": float(np.percentile(samples, 95)),
            "warmup": warmup, "repeats": repeats, "batch": 1, "imgsz": list(image.shape[-2:]),
            "precision": "FP32", "device": str(image.device), "fusion": False, "scope": "raw model forward"}

In [ ]:
with checked_stage("dataset_validation", validation=True):
    DATASET = prepare_validation_dataset(DATA_YAML, DATASET_ROOT, RUN_DIR) if RUN_FULL_VALIDATION else None
    if DATASET is None:
        reason = "RUN_FULL_VALIDATION=False" if not RUN_FULL_VALIDATION else "DATA_YAML tidak tersedia; mAP belum dievaluasi."
        STATE["validation_status"] = "NOT_RUN"
        STATE["metrics"]["validation"] = {"status": "NOT_RUN", "reason": reason}
        print(reason)
    else:
        assert_class_match(RELOADED.names, DATASET["info"]["names"])
        STATE["manifest"]["evaluation_dataset"] = {key: value for key, value in DATASET.items() if key != "info"}
        BEFORE["validation"] = evaluate_checkpoint(BASELINE_COMPARISON_PATH, DATASET, RUN_DIR, "baseline_val")
        AFTER["validation"] = evaluate_checkpoint(PRUNED_PATH, DATASET, RUN_DIR, "pruned_val")
        STATE["validation_status"] = "PASS"
        STATE["metrics"]["validation"] = {"status": "PASS", "same_dataset_and_protocol": True}

In [ ]:
with checked_stage("optional_cuda_latency", validation=True):
    if BENCHMARK_LATENCY:
        BEFORE["latency"] = cuda_latency(BASELINE_CORE, IMAGE, LATENCY_WARMUP, LATENCY_REPEATS)
        AFTER["latency"] = cuda_latency(RELOADED, IMAGE, LATENCY_WARMUP, LATENCY_REPEATS)
    else:
        BEFORE["latency"] = AFTER["latency"] = {"status": "NOT_RUN", "reason": "BENCHMARK_LATENCY=False"}

## 11 — Ringkasan Hasil dan Visualisasi
`structural_smoke_status` memerlukan seluruh syarat struktur, save/reload, dan inferensi.
`validation_status` hanya menjelaskan evaluasi dataset. PASS struktural tidak membuktikan mAP,
peningkatan latency, atau keberhasilan eksperimen final skripsi. Prediksi kosong pada input sintetis
tetap valid bila formatnya benar; tidak diklaim sebagai deteksi objek nyata.

In [ ]:
import matplotlib.pyplot as plt

with checked_stage("summary"):
    rows = []
    for label, metrics in (("baseline", BEFORE), ("pruned_reload", AFTER)):
        val = metrics.get("validation", {})
        rows.append({"model": label, "parameters": metrics["parameters"], "GFLOPs": metrics["gflops"],
                     "checkpoint_MiB_FP32": metrics["checkpoint_bytes"] / 1024**2,
                     "mAP50-95": val.get("map50_95"), "mAP50": val.get("map50"),
                     "precision": val.get("precision"), "recall": val.get("recall")})
    SUMMARY = pd.DataFrame(rows)
    display(SUMMARY)
    SUMMARY.to_csv(RUN_DIR / "before_after_summary.csv", index=False)
    figure, axes = plt.subplots(1, 2, figsize=(8, 3))
    for axis, key, title in zip(axes, ("parameters", "gflops"), ("Jumlah parameter", "GFLOPs (unfused)")):
        axis.bar(["Baseline", "Pruned"], [BEFORE[key], AFTER[key]], color=["#64748b", "#0f766e"])
        axis.set_title(title)
    figure.tight_layout()
    figure.savefig(RUN_DIR / "before_after.png", dpi=150)
    plt.show()
    print("structural_smoke_status:", STATE["structural_smoke_status"])
    print("validation_status:", STATE["validation_status"])
    print("Actual eligible-channel ratio:", STATE["manifest"]["actual_eligible_channel_ratio"])
    print("Actual parameter reduction:", STATE["manifest"]["parameter_reduction_ratio"])

## 12 — Reproducibility dan Artifact Output
Seluruh output tersimpan dalam folder run baru, termasuk laporan parsial saat terjadi error.
`pruned_fp32.pt` dan `baseline_fp32.pt` memakai format penyimpanan yang sama; ukuran artifact
sumber tersimpan terpisah di provenance. `RESTORE.md` dan `verify_reload.py` menjelaskan restore.
W&B mencatat ringkasan ketika tracking aktif; notebook tidak mengunggah checkpoint baru otomatis.

Referensi utama:
- Lee et al., [Layer-adaptive sparsity for the Magnitude-based Pruning, ICLR 2021](https://openreview.net/forum?id=H6ATjJ0TKdf)
  dan [kode asli](https://github.com/jaeho-lee/layer-adaptive-sparsity/blob/main/tools/pruners.py).
- [Torch-Pruning 1.6.1](https://github.com/VainF/Torch-Pruning/tree/v1.6.1),
  [LAMPImportance](https://github.com/VainF/Torch-Pruning/blob/v1.6.1/torch_pruning/pruner/importance.py),
  dan [contoh YOLOv8](https://github.com/VainF/Torch-Pruning/blob/v1.6.1/examples/yolov8/yolov8_pruning.py).
  Contoh YOLOv8 memuat rewrite modul dan fine-tuning; keduanya **tidak diadopsi** di sini.
- [MP-YOLO](https://doi.org/10.1016/j.jvcir.2025.104560), khususnya §3.6: ranking LAMP dan
  perlindungan head. Paper juga memakai perubahan arsitektur dan fine-tuning; hasilnya bukan target smoke test ini.
- [ZZ-YOLOv11](https://doi.org/10.3390/s25113399), konteks kompresi kendaraan dan distilasi.
  Deskripsi sensitivitas/progressive recovery di paper tersebut tidak mengganti definisi LAMP asli di notebook ini.
- [Laporan teknis YOLO26](https://arxiv.org/abs/2606.03748), dibaca bersama source versi terpasang;
  YOLO26 tidak diasumsikan identik dengan YOLOv8.

In [ ]:
with checked_stage("final_reports"):
    flush_reports()
    if WB_RUN is not None:
        WB_RUN.log({"smoke/structural_pass": int(STATE["structural_smoke_status"] == "PASS"),
                    "smoke/validation_ran": int(STATE["validation_status"] == "PASS"),
                    "pruning/actual_channel_ratio": STATE["manifest"]["actual_eligible_channel_ratio"],
                    "baseline/parameters": BEFORE["parameters"], "pruned/parameters": AFTER["parameters"],
                    "baseline/gflops": BEFORE["gflops"], "pruned/gflops": AFTER["gflops"]})
        WB_RUN.summary.update({"structural_smoke_status": STATE["structural_smoke_status"],
                               "validation_status": STATE["validation_status"],
                               "checkpoint_sha256": STATE["provenance"]["sha256"]})
        WB_RUN.finish()
    print("Output:", RUN_DIR)
    display(pd.DataFrame([{"file": path.name, "bytes": path.stat().st_size}
                          for path in sorted(RUN_DIR.iterdir()) if path.is_file()]))